# Lab3.4 — DMA pentru dimensiuni diferite de transfer

Într-un sistem Edge, datele transferate între PS și PL pot varia de la câțiva bytes la cadre video de sute de kilobytes. În acest task observăm cum se modifică timpul end-to-end în funcție de dimensiunea transferului.


## Obiective

- realizarea repetată a transferurilor DMA;
- măsurarea timpului end-to-end;
- observarea overhead-ului fix al unei tranzacții DMA;
- calculul unui throughput efectiv preliminar.


## 1. Inițializare


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

BITSTREAM_DIR = Path("bitstream")
BITSTREAM_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/onigas/PYNQ_Workshop/master/Session_4/bitstream"
FILES = ("dma_tutorial.bit", "dma_tutorial.hwh")

for filename in FILES:
    local_path = BITSTREAM_DIR / filename
    if not local_path.exists():
        print(f"Downloading {filename} ...")
        try:
            urlretrieve(f"{BASE_URL}/{filename}", local_path)
        except Exception as e:
            raise RuntimeError(
                f"Nu s-a putut descărca {filename}. "
                "Copiați manual fișierele dma_tutorial.bit și dma_tutorial.hwh "
                "în directorul Lab3/bitstream."
            ) from e

print("DMA overlay files are ready.")

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

overlay = Overlay("bitstream/dma_tutorial.bit")
dma_send = overlay.axi_dma_from_ps_to_pl
dma_recv = overlay.axi_dma_from_pl_to_ps

## 2. Funcție de măsurare

Măsurarea începe înainte de lansarea canalelor DMA și se termină după finalizarea ambelor canale. Alocarea bufferelor nu este inclusă în timp.


In [ ]:
def dma_loopback_once(n_elements):
    inp = allocate(shape=(n_elements,), dtype=np.uint32)
    out = allocate(shape=(n_elements,), dtype=np.uint32)

    inp[:] = np.arange(n_elements, dtype=np.uint32)
    out[:] = 0

    if hasattr(inp, "flush"):
        inp.flush()

    t0 = time.perf_counter()
    dma_recv.recvchannel.transfer(out)
    dma_send.sendchannel.transfer(inp)
    dma_send.sendchannel.wait()
    dma_recv.recvchannel.wait()
    t1 = time.perf_counter()

    if hasattr(out, "invalidate"):
        out.invalidate()

    ok = np.array_equal(inp, out)
    elapsed = t1 - t0
    nbytes = inp.nbytes

    del inp, out
    return nbytes, elapsed, ok

## 3. Test pentru mai multe dimensiuni


In [ ]:
sizes = (16, 64, 256, 1024, 4096, 16384, 65536)
results = []

for n in sizes:
    nbytes, elapsed, ok = dma_loopback_once(n)
    throughput = nbytes / elapsed / 1e6
    results.append((n, nbytes, elapsed, throughput, ok))
    print(
        f"{n:7d} elements | {nbytes:8d} B | "
        f"{elapsed*1e3:9.3f} ms | {throughput:9.3f} MB/s | ok={ok}"
    )

## 4. Interpretare

Pentru transferuri mici, timpul total este puternic influențat de costurile fixe ale lansării și sincronizării DMA. Pentru transferuri mai mari, costul fix este amortizat pe un volum mai mare de date.

Throughput-ul calculat mai sus este definit ca:

```text
payload throughput = dimensiunea bufferului / timpul end-to-end
```

Deși datele traversează atât DMA-ul de trimitere cât și pe cel de recepție, considerăm un singur payload procesat end-to-end.


## 5. Exercițiu

Repetați fiecare dimensiune de **3 ori** și calculați timpul mediu. Comparați stabilitatea măsurării pentru buffere mici și mari.


In [ ]:
# Scrieți aici soluția


## 6. Întrebări

1. De ce throughput-ul este de obicei mai mic pentru transferurile foarte scurte?
2. Ce costuri fixe sunt incluse în măsurarea end-to-end?
3. De ce nu includem alocarea bufferului în timpul DMA?
4. Cum s-ar modifica experimentul dacă blocul din PL ar efectua și o operație de calcul?
